In [40]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

import kagglehub
from kagglehub import KaggleDatasetAdapter

In [41]:
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "mssmartypants/paris-housing-price-prediction",
    "ParisHousing.csv",
)

/tmp/ipykernel_1794/1695127005.py:1: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'paris-housing-price-prediction' dataset.


In [42]:
X = df.drop('price', axis=1)
y = df["price"]

# X = X.fillna(0)

In [43]:
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten()

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

In [45]:
learning_rate = 0.1
epochs = 2000
hidden_layer_size = 64

In [46]:
class NeuralNetwork:
    def __init__(self, input_size):
        self.a1 = np.random.randn(input_size, hidden_layer_size) * 0.1
        self.b1 = np.zeros((1, hidden_layer_size))
        self.a2 = np.random.randn(hidden_layer_size, 1) * 0.1
        self.b2 = np.zeros((1, 1))

    def forward(self, X):
        self.linear_output_1 = np.dot(X, self.a1) + self.b1
        self.relu_output = np.maximum(0, self.linear_output_1)
        self.output_layer_input = np.dot(self.relu_output, self.a2) + self.b2
        return self.output_layer_input.flatten()

    def backward(self, X, y):
        error = (self.output_layer_input - y.reshape(-1, 1)) / X.shape[0]
        da2 = np.dot(self.relu_output.T, error)
        db2 = np.sum(error, axis=0, keepdims=True)

        d_relu_output = np.dot(error, self.a2.T)
        d_linear_output_1 = d_relu_output * (self.linear_output_1 > 0)
        da1 = np.dot(X.T, d_linear_output_1)
        db1 = np.sum(d_linear_output_1, axis=0, keepdims=True)

        self.a1 -= learning_rate * da1
        self.b1 -= learning_rate * db1
        self.a2 -= learning_rate * da2
        self.b2 -= learning_rate * db2

    def fit(self, X, y):
        for epoch in range(epochs):
            self.forward(X)
            self.backward(X, y)

    def predict(self, X):
        return self.forward(X)

In [47]:
model = NeuralNetwork(X_train.shape[1])
model.fit(X_train, y_train)

In [48]:
y_pred_scaled = model.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

In [49]:
print(f"RMSE: ${np.sqrt(mean_squared_error(y_true, y_pred)):.2f}")
print(f"R2: {r2_score(y_true, y_pred):.4f}")
print(f"MAPE: {mean_absolute_percentage_error(y_true, y_pred):.4f}")

RMSE: $108841.36
R2: 0.9986
MAPE: 0.0658


In [50]:
# Классификация

In [51]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler

In [52]:
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "praveengovi/credit-risk-classification-dataset",
    "customer_data.csv",
)

/tmp/ipykernel_1794/1867492656.py:1: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'credit-risk-classification-dataset' dataset.


In [53]:
X = df.drop(['label', "id"], axis=1)
y = df["label"]

In [54]:
X = X.fillna(0)

In [55]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [56]:
scaler_classifier = StandardScaler()
x_train_scaled = scaler_classifier.fit_transform(x_train)
x_test_scaled = scaler_classifier.transform(x_test)

In [57]:
clf = LogisticRegression(random_state=0, max_iter=1000, class_weight="balanced").fit(x_train_scaled, y_train)

In [58]:
y_pred = clf.predict(x_test_scaled)

In [59]:
print(f"Precision: {precision_score(y_test, y_pred)}")
print(f"Recall: {recall_score(y_test, y_pred)}")
print(f"F1_score: {f1_score(y_test, y_pred)}")

Precision: 0.319672131147541
Recall: 0.7358490566037735
F1_score: 0.44571428571428573


In [60]:
X = df.drop(['label', "id"], axis=1)
y = df["label"]

In [61]:
X = X.fillna(0)

In [62]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)   # нормализуем train
X_test_norm  = scaler.transform(X_test)        # нормализуем test

model = LinearSVC(max_iter=10000, class_weight="balanced")
model.fit(X_train_norm, y_train)

predictions = model.predict(X_test_norm)             # жёсткие классы 0/1

print("precision:", precision_score(y_test, predictions))
print("recall:   ", recall_score(y_test, predictions))
print("f1:       ", f1_score(y_test, predictions))

precision: 0.30833333333333335
recall:    0.6981132075471698
f1:        0.4277456647398844


In [76]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model_tf = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_norm.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model_tf.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

model_tf.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

In [77]:
history = model_tf.fit(
    X_train_norm,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_norm, y_test),
    verbose=1
)

Epoch 1/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.7389 - loss: 0.5844 - precision_5: 0.2162 - recall_5: 0.1395 - val_accuracy: 0.7644 - val_loss: 0.5341 - val_precision_5: 0.0000e+00 - val_recall_5: 0.0000e+00
Epoch 2/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8089 - loss: 0.5151 - precision_5: 0.5000 - recall_5: 0.0058 - val_accuracy: 0.7644 - val_loss: 0.5386 - val_precision_5: 0.0000e+00 - val_recall_5: 0.0000e+00
Epoch 3/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8078 - loss: 0.5031 - precision_5: 0.0000e+00 - recall_5: 0.0000e+00 - val_accuracy: 0.7644 - val_loss: 0.5353 - val_precision_5: 0.0000e+00 - val_recall_5: 0.0000e+00
Epoch 4/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8100 - loss: 0.5046 - precision_5: 0.6667 - recall_5: 0.0116 - val_accuracy: 0.7644 - val_loss: 0.5349 - val_precision_5: 0.0000e+00 - val_recall_5: 0.0000e+00
Epoch 5/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8078 - loss: 0.4933 - precision

In [78]:
y_pred_proba_tf = model_tf.predict(X_test_norm)

y_pred_tf = (y_pred_proba_tf > 0.1).astype(int)

precision_tf = precision_score(y_test, y_pred_tf)
recall_tf = recall_score(y_test, y_pred_tf)
f1_tf = f1_score(y_test, y_pred_tf)

print(f"Precision: {precision_tf:.4f}")
print(f"Recall:    {recall_tf:.4f}")
print(f"F1-score:  {f1_tf:.4f}")


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
Precision: 0.2757
Recall:    0.9623
F1-score:  0.4286
